# Objective

Fine-tune Qwen/Qwen2.5-1.5B-Instruct for structured tool/function calling using QLoRA and supervised fine-tuning on a representative subset of the Salesforce/xlam-function-calling-60k dataset.

In [ ]:
import json
import os
from pathlib import Path

import torch
import mlflow

from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
)

from trl import SFTConfig, SFTTrainer

In [ ]:
# ============================================================
# CELL 2: LOAD REPRESENTATIVE DATASETS
# ============================================================

from datasets import load_from_disk

DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "representative_qlora"
)

TRAIN_PATH = os.path.join(DATA_DIR, "train")
VALIDATION_PATH = os.path.join(DATA_DIR, "validation")
TEST_PATH = os.path.join(DATA_DIR, "test")

print("=" * 70)
print("LOADING REPRESENTATIVE DATASETS")
print("=" * 70)

train_dataset = load_from_disk(TRAIN_PATH)
validation_dataset = load_from_disk(VALIDATION_PATH)
test_dataset = load_from_disk(TEST_PATH)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(validation_dataset))
print("Test samples:", len(test_dataset))

print("\nColumns:")
print(train_dataset.column_names)

print("\nComplexity distribution:")
print(train_dataset["complexity"][:10])

In [ ]:
# ============================================================
# CELL 3: FORMAT DATA FOR SFT TRAINING
# ============================================================

import json

def format_tool_calling_example(example):

    tools = json.loads(example["tools"])
    answers = json.loads(example["answers"])

    system_message = (
        "You are a helpful assistant. "
        "You have access to the following tools.\n"
        "# Tools\n"
        "<tools>\n"
        + json.dumps(tools, ensure_ascii=False)
        + "\n</tools>"
    )

    prompt = (
        "<|im_start|>system\n"
        + system_message
        + "\n<|im_end|>\n"
        "<|im_start|>user\n"
        + example["query"]
        + "\n<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    completion_parts = []

    for answer in answers:

        tool_call = (
            "<tool_call>\n"
            + json.dumps(answer, ensure_ascii=False, separators=(",", ":"))
            + "\n</tool_call>"
        )

        completion_parts.append(tool_call)

    completion = "\n".join(completion_parts)

    return {
        "prompt": prompt,
        "completion": completion
    }


train_sft = train_dataset.map(
    format_tool_calling_example,
    remove_columns=train_dataset.column_names
)

validation_sft = validation_dataset.map(
    format_tool_calling_example,
    remove_columns=validation_dataset.column_names
)

print("=" * 70)
print("SFT DATASET")
print("=" * 70)

print("Train:", len(train_sft))
print("Validation:", len(validation_sft))

print("\nExample:")
print(train_sft[0]["prompt"])
print(train_sft[0]["completion"])

# Load Qwen Model in 4-bit

We load the pretrained Qwen/Qwen2.5-1.5B-Instruct model using 4-bit NF4 quantization.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

print("4-bit model loaded successfully.")
print(f"Model device: {model.device}")

# Configure LoRA Adapters

LoRA adds trainable low-rank adapter matrices to selected attention projection layers while keeping the pretrained model weights frozen.

We use rank 16, scaling factor 32, 5% dropout, and target the query, key, value, and output projections.

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

print("LoRA configuration created.")
print(lora_config)

In [ ]:
# Configure Supervised Fine-Tuning

The model is trained using supervised fine-tuning on the formatted prompt/completion dataset.

The training configuration is designed for the available T4 GPU while maintaining a controlled experiment.

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=str(CHECKPOINTS_DIR / "qlora_representative_4000"),

    num_train_epochs=3,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    logging_steps=50,

    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    lr_scheduler_type="cosine",

    optim="paged_adamw_8bit",

    fp16=False,
    bf16=False,

    gradient_checkpointing=True,

    report_to="mlflow",

    seed=42,

    max_length=1024,
    completion_only_loss=True,
    packing=False,
)

print("SFT configuration created.")
print(training_args)

# Train and Save the QLoRA Adapter

The SFT trainer combines the prepared dataset, quantized Qwen base model, and LoRA configuration

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

# Keep trainable LoRA parameters in FP32 for stable training on T4.
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

print("Trainable parameters:")
trainer.model.print_trainable_parameters()

print("\nStarting QLoRA training...")
trainer.train()

FINAL_CHECKPOINT = CHECKPOINTS_DIR / "qlora_representative_4000_final"

trainer.save_model(str(FINAL_CHECKPOINT))
tokenizer.save_pretrained(str(FINAL_CHECKPOINT))

print(f"\nQLoRA adapter saved to: {FINAL_CHECKPOINT}")